In [324]:
import random
import time
from IPython.display import clear_output
import numpy as np

N_ROW = 6
N_COL = 8

BASE = 0            # → .normal
C1 = 1              # → .corridor-1 ↰↳
C2 = 2              # → .corridor-2 ↲↱
N_HOLE = 3          # → .near_hole
HOLE = 4            # → .hole

N_WUMP = 10
BAT = 20
WUMPUS = 40
PLAYER = 70

TRIG_BAT = 100

#  0 = grille de base       → .normal
#  1 = couloir 1 ↰↳         → .corridor-1
#  2 = couloir 2 ↲↱         → .corridor-2
#  3 = proche du puit       → .near_hole
#  4 = puit                 → .hole
#
# 10 = close to the wumpus  → .near_wumpus
# 11 = 
# 40 = joueur               → .player



In [325]:
EASY = 0
NORMAL = 1
HARD = 2

In [ ]:
matrix = [[0] * N_COL for _ in range(N_ROW)]

def generate_grid(diff) :
    match diff :
        case 0 :
            difficulty = [0]*40 + [random.choice([1, 2]) for _ in range(14)]
        case 1 :
            difficulty = [0]*30 + [random.choice([1, 2]) for _ in range(24)]
        case 2 :
            difficulty = [0]*20 + [random.choice([1, 2]) for _ in range(34)]

    # North
    col = 0

    while col < N_COL :
        matrix[0][col] = difficulty.pop(random.randint(0, len(difficulty)-1))
        col = col+1
    
    # West
    row = 0

    while row < N_ROW :
        matrix[row][0] = difficulty.pop(random.randint(0, len(difficulty)-1))
        row = row+1

    # Center
    row = 1

    while row < N_ROW-1:
        col = 1
        while col < len(matrix[row])-1 :
            rand = random.randint(0, len(difficulty)-1)
            while check_nw(row, col, difficulty[rand]) :
                rand = random.randint(0, len(difficulty)-1)

            matrix[row][col] = difficulty.pop(rand)
            col = col+1
        row = row+1

    # East
    row = 1
    col = N_COL-1

    while row < N_ROW-1 :
        rand = random.randint(0, len(difficulty)-1)
        while check_nw(row, col, difficulty[rand]) | check_ne(row, col, difficulty[rand]) :
            rand = random.randint(0, len(difficulty)-1)
        matrix[row][col] = difficulty.pop(rand)
        row = row+1

    # South
    row = N_ROW-1
    col = 1

    while col < N_COL-1 :
        rand = random.randint(0, len(difficulty)-1)
        while check_nw(row, col, difficulty[rand]) | check_sw(row, col, difficulty[rand]) :
            rand = random.randint(0, len(difficulty)-1)
        matrix[row][col] = difficulty.pop(rand)
        col = col+1

    # Corner
    rand = random.randint(0, len(difficulty)-1)
    stop_while = 0
    while (check_corner(difficulty[rand])) and stop_while < 5 :
        rand = random.randint(0, len(difficulty)-1)
        stop_while = stop_while+1
    if stop_while == 5 :
        matrix[row][col] = 0
    else :
        matrix[row][col] = difficulty.pop(rand)
    col = col+1

    return matrix

# =============================
#           Check

def check_nw(row, col, type) :
    retry = False
    if type == 2 :
        if matrix[row-1][col] == 1 :
            if matrix[row][col-1] == 1 :
                if (matrix[row-1][col-1] == 2) :
                    retry = True
    return retry

def check_ne(row, col, type) :
    retry = False
    if type == 1 :
        if matrix[row-1][col] == 2 :
            if matrix[row][0] == 2 :
                if (matrix[row-1][0] == 1) :
                    retry = True
    return retry

def check_sw(row, col, type) :
    retry = False
    if type == 1 :
        if matrix[0][col] == 2 :
            if matrix[row][col-1] == 2 :
                if (matrix[0][col-1] == 1) :
                    retry = True
    return retry

def check_se(row, col, type) :
    retry = False
    if type == 2 :
        if matrix[0][col] == 1 :
            if matrix[row][0] == 1 :
                if (matrix[0][0] == 2) :
                    retry = True
    return retry

def check_corner(type) :
    row = N_ROW-1
    col = N_COL-1

    retry = False
    match type :
        case 1 :
            retry = check_ne(row, col, type) | check_sw(row, col, type)
        case 2 :
            retry = check_nw(row, col, type) | check_se(row, col, type)
    return retry

In [ ]:
# par IA
def print_forbidden_patterns(matrix):
    n_row = len(matrix)
    n_col = len(matrix[0])

    found = False

    for r in range(n_row):
        for c in range(n_col):
            positions = [
                (r, c),
                (r, (c + 1) % n_col),
                ((r + 1) % n_row, c),
                ((r + 1) % n_row, (c + 1) % n_col),
            ]

            values = [
                matrix[r][c],
                matrix[r][(c + 1) % n_col],
                matrix[(r + 1) % n_row][c],
                matrix[(r + 1) % n_row][(c + 1) % n_col],
            ]

            if values == [2, 1, 1, 2]:
                found = True
                print("❌ Pattern interdit détecté aux positions :")
                print(f"  (r,c)   = {positions[0]} → {values[0]}")
                print(f"  (r,c+1) = {positions[1]} → {values[1]}")
                print(f"  (r+1,c) = {positions[2]} → {values[2]}")
                print(f"  (r+1,c+1) = {positions[3]} → {values[3]}")
                print()
                time.sleep(5)

    if not found:
        print("✅ Aucun pattern interdit trouvé")
    return found


In [350]:

for _ in range(4000) :
    clear_output()
    grid = generate_grid(HARD)
    for row in grid:
        for col in row:
            print(f"{col:2} ", end="")
        print()
    if print_forbidden_patterns(grid) :
        break

 0  2  0  2  2  0  2  0 
 0  2  2  1  1  0  0  1 
 2  1  1  0  1  0  2  1 
 1  1  0  1  2  0  1  0 
 2  0  2  2  0  2  2  1 
 1  2  0  1  1  0  1  0 
✅ Aucun pattern interdit trouvé


In [329]:
mat = grid
mat[0][0] = 6
display(mat)
display(grid)

[[6, 2, 0, 1, 0, 2, 2, 0],
 [1, 2, 1, 2, 0, 1, 0, 1],
 [0, 1, 1, 2, 0, 0, 0, 1],
 [2, 0, 2, 0, 0, 2, 0, 1],
 [1, 2, 0, 1, 1, 1, 0, 1],
 [1, 1, 2, 1, 2, 0, 2, 1]]

[[6, 2, 0, 1, 0, 2, 2, 0],
 [1, 2, 1, 2, 0, 1, 0, 1],
 [0, 1, 1, 2, 0, 0, 0, 1],
 [2, 0, 2, 0, 0, 2, 0, 1],
 [1, 2, 0, 1, 1, 1, 0, 1],
 [1, 1, 2, 1, 2, 0, 2, 1]]

In [ ]:
move = ((-1, 0), (1, 0), (0, 1), (0, -1))
for i in range(4) :
    my = move[i%4][0]
    mx = move[i%4][1]
    print(mx)
    print(my)
    print